## Membership Inference Attacks

A **membership inference attack (MIA)** asks: given a trained generative model (or the synthetic data it released) and a candidate record $x$, can an adversary tell whether $x$ was in the **training set**?

If the answer is reliably yes, the synthesizer has leaked something about specific people, not just the population distribution. That is a privacy failure even when the released table looks anonymous.

In this notebook the **target** is the TabDDPM (ClavaDDPM) model trained in `implementations/tabular_data/single_table`. We use:

- **members**: real training rows (`data/trans.csv`)
- **non-members**: real holdout rows (`data/trans_holdout.csv`)
- **released synthetic table**: `results/single_table_synthesizing/trans/_final/trans_synthetic.csv`

The adversary's job is to label each candidate as member (`1`) or non-member (`0`). Random guessing on a balanced set scores **0.5 AUC**. Anything clearly above 0.5 means the synthetic release is more similar to training records than to held-out records from the same distribution.

### Setting

Two standard threat models differ in what the adversary can see.

#### Black-box

The adversary **does not** get model weights, gradients, or training loss. Typical access is one or both of:

- a **released synthetic dataset** $S$ (the usual case after publishing a table)
- a **sampling API**: query the generator for more synthetic rows, but not internals

A simple and widely used score for tabular synthetic data is **distance to closest record (DCR)**:

$$\mathrm{DCR}(x, S) = \min_{s \in S} \|x - s\|$$

If the model memorized training points, members tend to sit closer to $S$ than holdout points. The attack is then:

$$\hat{y}(x) = \mathbf{1}\{\mathrm{DCR}(x, S) < \tau\}$$

This is the attack view of the same nearest-neighbor idea used as a privacy *metric* in `privacy_evaluation_pipeline.ipynb`. The metric reports how often synthetic points hug the train set; the attack uses that proximity to classify *people*.


<div align="center">
  <img src="../images/MIA.png" alt="MIA" width="600" height="300">
</div>




Other black-box variants (not implemented here) include shadow-model attacks (train copies of the generator on auxiliary data, then a classifier on their outputs) and density-ratio attacks such as DOMIAS.

#### White-box

The adversary **does** get the trained generator (checkpoint, architecture, and preprocessing). For a diffusion model such as TabDDPM, a typical score is the **per-record denoising loss**: add noise to $x$ at random timesteps and measure how well the network predicts that noise. Overfit members usually have **lower** loss than holdout records, so the attack is:

$$\hat{y}(x) = \mathbf{1}\{\mathcal{L}_{\mathrm{diffusion}}(x) < \tau\}$$

White-box attacks are often stronger because they use the model's own likelihood/loss rather than a geometric proxy on samples. They also require the checkpoint (`results/models/...`), not only the published table. This notebook implements the **black-box DCR attack** on the released synthetic data, which matches a realistic “the table was published” setting.

### Simple black-box MIA example

Pipeline below:

1. Load train, holdout, and synthetic tables from the single-table run.
2. Encode categoricals and scale numerics the same way as the distance-based privacy metrics.
3. Build a balanced candidate set (equal members and non-members).
4. Score each candidate by **negative DCR** to the synthetic table (higher = more likely member).
5. Report ROC-AUC, TPR at a low FPR, and accuracy at a threshold fit on a calibration split.

In [1]:
from pathlib import Path
import os

def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")

set_project_root()

from logging import INFO
import json

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

from midst_toolkit.common.logger import log
from midst_toolkit.evaluation.privacy.distance_preprocess import preprocess_for_distance_computation

/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the single-table artifacts

These paths match the training / synthesizing notebooks under `implementations/tabular_data/single_table`.

In [2]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data" / "single_table"
base_data_dir = IMPLEMENTATION_ROOT / "data"
base_output_dir = IMPLEMENTATION_ROOT / "results"
TABLE_NAME = "trans"

real_train_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}.csv")
real_holdout_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}_holdout.csv")
synthetic_data = pd.read_csv(
    base_output_dir / "single_table_synthesizing" / TABLE_NAME / "_final" / f"{TABLE_NAME}_synthetic.csv"
)

with open(base_data_dir / "meta_info.json", "r") as f:
    meta_info = json.load(f)

log(INFO, f"Loaded {len(real_train_data)} training (member) rows")
log(INFO, f"Loaded {len(real_holdout_data)} holdout (non-member) rows")
log(INFO, f"Loaded {len(synthetic_data)} synthetic rows")

INFO :      Loaded 16000 training (member) rows
INFO :      Loaded 4000 holdout (non-member) rows
INFO :      Loaded 3200 synthetic rows


## Black-box DCR attack

Preprocessing follows the distance-based privacy metrics: one-hot categoricals and range-normalize numerics so L2 is comparable across columns.

We sample a **balanced** candidate set so chance-level accuracy is 50%. Scores are computed only against the synthetic table — the adversary never looks at model weights.

In [3]:
def min_l2_distance(query: torch.Tensor, reference: torch.Tensor, batch_size: int = 1024) -> np.ndarray:
    """DCR: L2 distance from each query row to its nearest synthetic row."""
    mins = []
    for start in range(0, query.size(0), batch_size):
        batch = query[start : start + batch_size]
        mins.append(torch.cdist(batch, reference, p=2).min(dim=1).values)
    return torch.cat(mins).cpu().numpy()


RNG = np.random.default_rng(42)

distance_train, distance_synthetic, distance_holdout = preprocess_for_distance_computation(
    meta_info, real_train_data, synthetic_data, real_holdout_data
)

n_candidates = min(len(distance_train), len(distance_holdout))
member_idx = RNG.choice(len(distance_train), size=n_candidates, replace=False)
non_member_idx = RNG.choice(len(distance_holdout), size=n_candidates, replace=False)

members = distance_train.iloc[member_idx].reset_index(drop=True)
non_members = distance_holdout.iloc[non_member_idx].reset_index(drop=True)
candidates = pd.concat([members, non_members], ignore_index=True)
labels = np.concatenate([np.ones(n_candidates, dtype=int), np.zeros(n_candidates, dtype=int)])

candidate_tensor = torch.tensor(candidates.to_numpy(), dtype=torch.float32)
synthetic_tensor = torch.tensor(distance_synthetic.to_numpy(), dtype=torch.float32)
dcr = min_l2_distance(candidate_tensor, synthetic_tensor)

# Higher score => predicted member. Memorization => members have smaller DCR.
mia_scores = -dcr

log(INFO, f"Attack set: {n_candidates} members + {n_candidates} non-members")
log(INFO, f"Mean DCR members:     {dcr[labels == 1].mean():.4f}")
log(INFO, f"Mean DCR non-members: {dcr[labels == 0].mean():.4f}")

INFO :      Attack set: 4000 members + 4000 non-members
INFO :      Mean DCR members:     1.2668
INFO :      Mean DCR non-members: 1.2802


## Evaluate the attack

- **ROC-AUC**: threshold-free ranking quality. 0.5 is chance; 1.0 is perfect membership detection.
- **TPR at 10% FPR**: how often we correctly flag members while wrongly accusing only 10% of non-members — closer to a high-confidence privacy risk than accuracy alone.
- **Accuracy**: threshold $\tau$ chosen on a calibration split with Youden's $J$ (maximize TPR $-$ FPR), then applied to the remaining test split.

In [4]:
auc = roc_auc_score(labels, mia_scores)
fpr, tpr, _ = roc_curve(labels, mia_scores)
idx_at_10_fpr = np.where(fpr <= 0.10)[0]
tpr_at_10_fpr = float(tpr[idx_at_10_fpr[-1]]) if len(idx_at_10_fpr) else 0.0

perm = RNG.permutation(len(labels))
split = len(labels) // 2
cal_idx, test_idx = perm[:split], perm[split:]

cal_scores, cal_labels = mia_scores[cal_idx], labels[cal_idx]
test_scores, test_labels = mia_scores[test_idx], labels[test_idx]

# Youden's J on the calibration split: maximize TPR - FPR.
cal_fpr, cal_tpr, cal_thresholds = roc_curve(cal_labels, cal_scores)
best_tau = cal_thresholds[np.argmax(cal_tpr - cal_fpr)]
test_preds = (test_scores >= best_tau).astype(int)
test_acc = accuracy_score(test_labels, test_preds)

log(INFO, f"ROC-AUC:           {auc:.4f}")
log(INFO, f"TPR @ 10% FPR:     {tpr_at_10_fpr:.4f}")
log(INFO, f"Calibrated tau:    {best_tau:.4f}  (applied to -DCR)")
log(INFO, f"Test accuracy:     {test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(dcr[labels == 1], bins=40, alpha=0.6, label="members (train)")
axes[0].hist(dcr[labels == 0], bins=40, alpha=0.6, label="non-members (holdout)")
axes[0].set_xlabel("DCR to synthetic table")
axes[0].set_ylabel("count")
axes[0].set_title("Black-box score: distance to closest synthetic row")
axes[0].legend()

axes[1].plot(fpr, tpr, label=f"DCR MIA (AUC={auc:.3f})")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="chance")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("Membership inference ROC")
axes[1].legend()

fig.tight_layout()
plt.show()

INFO :      ROC-AUC:           0.5066
INFO :      TPR @ 10% FPR:     0.0955
INFO :      Calibrated tau:    -0.9460  (applied to -DCR)
INFO :      Test accuracy:     0.5092
/var/folders/nr/v0nr1chn2vn21ctmvndcpsqc0000gq/T/ipykernel_7455/224653567.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## How to read the numbers

- **AUC near 0.5**: the synthetic table is about as close to holdout rows as to training rows. This black-box attack does not extract membership.
- **AUC well above 0.5**, or a large gap with **members having smaller DCR**: the generator is hugging its training set. Publishing $S$ lets an adversary rank likely members.
- A **weak attack here is not a privacy proof**. Stronger black-box attacks (shadow models, density ratios) or a white-box loss attack on the TabDDPM checkpoint can still succeed. Use this notebook as a first, interpretable check alongside the DCR / NNDR metrics in the privacy evaluation pipeline.